In [1]:
# Should be first to load all the env vars for browser
import os
from dotenv import load_dotenv

from browser import ChromeBrowser
from generalist.models.core import LLMBrowserWithTools, MLFlowLLMWrapper

load_dotenv()

assert os.getenv("CHROME_USER_DATA_DIR", None)
chrome_browser = ChromeBrowser()

/Users/maksim.rostov/pdev/freelectron/free-generalist/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/maksim.rostov/pdev/freelectron/free-generalist/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
sh: pidof: command not found


In [2]:
import mlflow
import logging

mlflow.langchain.autolog()   # this is needed to register traces within the experiment
experiment_name = f"agent_tool_test"
mlflow.set_experiment(experiment_name)
logging.getLogger().setLevel(logging.INFO)

llm_raw = LLMBrowserWithTools(chrome_browser)
llm = MLFlowLLMWrapper(llm_instance=llm_raw)

/Users/maksim.rostov/pdev/freelectron/free-generalist/.venv/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [3]:
text = ""

In [4]:
from generalist.agents.core import  AgentCodeWriterExecutor
from generalist.tools.data_model import Message

content_resources = Message(
    provided_by= "user",
    content=text,
    link= "",
)

cwd = "~/pdev/freelectron/free-generalist/src"
task = f"Write code that accepts an html and outputs markdown - USE crawl4ai ! Write the code into {cwd}/tools/parser.py (cwd is {cwd} and you should use tools)."
agent = AgentCodeWriterExecutor(activity=task, llm=llm)
out = agent.run([content_resources])


2026-04-23 12:31:28,179 - browser.llm_browser - call:105 - INFO - Queue for calling LLM's with long message window: [('Claude', 3), ('DeepSeek', 3), ('Mistral', 3), ('Qwen', 3)]
2026-04-23 12:31:28,180 - browser.llm_browser - _try_sessions:84 - INFO - Trying Gemini ...
2026-04-23 12:31:50,318 - generalist.agents.workflows.tasks.plan_action - plan_next_action:42 - INFO - Plan: Gemini zei
I have gathered the following information:
Crawl4AI Basics: It uses an asynchronous crawler (AsyncWebCrawler) and provides high-quality Markdown by default through result.markdown.
Specific Conversion: While it's primarily a crawler, it can process raw HTML if passed into specific strategies, or it can be used to "crawl" a URL and return the Markdown directly.
File Path: The target file is ~/pdev/freelectron/free-generalist/src/tools/parser.py.
What is still missing:
Confirmation of the exact function signature needed (e.g., just html_to_markdown(html_content: str)).
Handling the "input is HTML string" 

In [5]:
# print(agent.agent_state["context"])#[-1].content)
print(agent.agent_state)

{'task': 'Write code that accepts an html and outputs markdown - USE crawl4ai ! Write the code into ~/pdev/freelectron/free-generalist/src/tools/parser.py (cwd is ~/pdev/freelectron/free-generalist/src and you should use tools).', 'plan': 'Gemini zei\nI have gathered the following information:\nCrawl4AI Basics: It uses an asynchronous crawler (AsyncWebCrawler) and provides high-quality Markdown by default through result.markdown.\nSpecific Conversion: While it\'s primarily a crawler, it can process raw HTML if passed into specific strategies, or it can be used to "crawl" a URL and return the Markdown directly.\nFile Path: The target file is ~/pdev/freelectron/free-generalist/src/tools/parser.py.\nWhat is still missing:\nConfirmation of the exact function signature needed (e.g., just html_to_markdown(html_content: str)).\nHandling the "input is HTML string" case: Most Crawl4AI examples use a URL. I need to ensure the code correctly uses Crawl4AI\'s internal generators or a "data URI/loc